In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# 1. OpenML에서 실제 은행 마케팅(Bank Marketing) 데이터 로드 (몇 초 소요될 수 있음)
print("대규모 마케팅 데이터를 불러오는 중입니다...")
bank_data = fetch_openml(name='bank-marketing', version=1, as_frame=True, parser='auto')
df_bank = bank_data.frame

# 2. 전처리: 타겟 변수(가입 여부: yes/no -> 1/0) 변환 및 범주형 데이터 원핫인코딩
df_bank['Class'] = df_bank['Class'].map({'1': 0, '2': 1}) # 원본 데이터 라벨 맞춤
X = pd.get_dummies(df_bank.drop('Class', axis=1), drop_first=True)
y = df_bank['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. XGBoost 모델 학습
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_xgb.fit(X_train, y_train)

# 4. 리드 스코어링 (Lead Scoring): 가입할 확률 예측
lead_scores = model_xgb.predict_proba(X_test)[:, 1] # 1(가입)이 될 확률
df_results = X_test.copy()
df_results['가입_확률(Lead_Score)'] = lead_scores

# 확률이 높은 순으로 정렬하여 세일즈 팀에 넘겨줄 '초고관여 VIP 리스트' 추출
vip_leads = df_results.sort_values(by='가입_확률(Lead_Score)', ascending=False)

print("\n🎯 [BOFU: 전환 분석] 세일즈 팀에 전달할 '결제 확률 최상위 VIP 고객' 리스트")
print(vip_leads[['가입_확률(Lead_Score)']].head(5))


대규모 마케팅 데이터를 불러오는 중입니다...

🎯 [BOFU: 전환 분석] 세일즈 팀에 전달할 '결제 확률 최상위 VIP 고객' 리스트
       가입_확률(Lead_Score)
44510           0.991508
44501           0.990702
43285           0.987880
24068           0.986723
43249           0.983613


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [01:35:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
